# Notebook 2 – CNN Architecture Design



In [1]:
import time
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.14.0+cpu | device: cpu


## Concept and explain it yourself
A CNN moves from pixels to learned features. Convolutions detect patterns; activations add nonlinearity; pooling or stride reduces spatial dimensions; BatchNorm can stabilize training; Dropout regularizes; fully connected layers map features to class logits. PyTorch inputs use `(N,C,H,W)`, and `CrossEntropyLoss` expects raw logits.

For a grayscale 8×8 digit the input is `(N,1,8,8)`. A 3×3 convolution with padding 1 preserves height/width; 2×2 pooling halves them. Explain why a deeper model can use more channels as spatial dimensions shrink.

In [2]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
d=load_digits(); x=torch.tensor(d.images,dtype=torch.float32).unsqueeze(1)/16
x=F.interpolate(x,(16,16),mode='bilinear',align_corners=False); y=torch.tensor(d.target,dtype=torch.long)
a,b,ya,yb=train_test_split(x,y,test_size=.2,random_state=42,stratify=y)
tr=DataLoader(TensorDataset(a,ya),batch_size=128,shuffle=True); va=DataLoader(TensorDataset(b,yb),batch_size=256)
class CNN(nn.Module):
 def __init__(self,width=16,depth=1):
  super().__init__(); layers=[]; c=1
  for _ in range(depth): layers += [nn.Conv2d(c,width,3,padding=1),nn.ReLU(),nn.MaxPool2d(2)]; c=width; width*=2
  self.features=nn.Sequential(*layers,nn.AdaptiveAvgPool2d((2,2))); self.head=nn.Linear(c*4,10)
 def forward(self,x): return self.head(self.features(x).flatten(1))
def trial(width,depth):
 torch.manual_seed(42)
 m=CNN(width,depth).to(device); opt=torch.optim.Adam(m.parameters(),lr=1e-3); start=time.perf_counter()
 for _ in range(2):
  m.train()
  for xb,y in tr:
   xb,y=xb.to(device),y.to(device); opt.zero_grad(); loss=nn.functional.cross_entropy(m(xb),y); loss.backward(); opt.step()
 m.eval(); good=total=0
 with torch.no_grad():
  for xb,y in va: good+=(m(xb.to(device)).argmax(1).cpu()==y).sum().item(); total+=len(y)
 return sum(p.numel() for p in m.parameters()),good/total,time.perf_counter()-start
for name,w,d in [('shallow',12,1),('medium',16,2),('deeper',16,3)]:
 p,acc,sec=trial(w,d); print(f'{name:8} params={p:,} val_acc={acc:.3f} train_s={sec:.2f}')

shallow  params=610 val_acc=0.100 train_s=0.46
medium   params=6,090 val_acc=0.294 train_s=0.84
deeper   params=25,866 val_acc=0.689 train_s=1.47


## Simple example and interpretation
Read the measurements from your own run; they vary by device. Prefer the least expensive model that meets the accuracy requirement. More depth can improve representation capacity, but can also add latency and overfit. Run several seeds before concluding a small accuracy difference is real.

**Check:** What changed between models? What would you hold constant in the next experiment?